In [ ]:
"""
NASA CMAPSS Turbofan Engine Degradation — SMA Framework Test
=============================================================
Core question: are velocity, direction, and acceleration of the
statistical parameter vector θ(t) predictive of Remaining Useful
Life (RUL)?

This is the key test: in an endogenous degradation system, does
directional drift in θ-space carry predictive signal?

Pipeline:
  1. Download + load CMAPSS (FD001–FD004)
  2. Build θ(t) per engine — rolling statistical features
  3. Compute kinematic features: v(t), s(t), a(t), D(t), d(t,W)
  4. Test feature importance: which features predict RUL best?
  5. Test directional persistence: is E[P(t,W)] > 0? (momentum = degradation)
  6. Visualise attractor geometry coloured by RUL
  7. Train a simple baseline model (gradient boosted tree) on
     raw sensors vs kinematic features — compare RMSE
"""

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import skew, kurtosis, ttest_1samp, pearsonr, spearmanr
from scipy.spatial.distance import cosine
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.inspection import permutation_importance
import urllib.request
import zipfile

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD CMAPSS
# ─────────────────────────────────────────────────────────────────────

DATA_DIR = "cmapss_data"
os.makedirs(DATA_DIR, exist_ok=True)

def download_cmapss():
    """Download NASA CMAPSS dataset."""
    url = ("https://data.nasa.gov/api/views/ff5v-kuh6/rows.csv"
           "?accessType=DOWNLOAD")
    # Direct zip mirror (more reliable)
    zip_url = ("https://raw.githubusercontent.com/schwingbach/"
               "deepl-rul-cmapss/master/CMAPSSData.zip")
    zip_path = os.path.join(DATA_DIR, "CMAPSSData.zip")

    if not os.path.exists(os.path.join(DATA_DIR, "train_FD001.txt")):
        print("  Downloading CMAPSS...")
        try:
            urllib.request.urlretrieve(zip_url, zip_path)
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(DATA_DIR)
            print("  Downloaded and extracted.")
        except Exception as e:
            print(f"  Download failed: {e}")
            print("  Manual download: go to https://data.nasa.gov/dataset/"
                  "Turbofan-Engine-Degradation-Simulation-Data-Set-1/vrks-gjie")
            print("  Place train_FD001.txt ... test_FD004.txt in cmapss_data/")
            return False
    return True

def load_cmapss(dataset="FD001"):
    """Load one CMAPSS sub-dataset."""
    cols = (["engine_id", "cycle"] +
            [f"op_{i}" for i in range(1,4)] +
            [f"s_{i}" for i in range(1,22)])

    train = pd.read_csv(
        os.path.join(DATA_DIR, f"train_{dataset}.txt"),
        sep=r'\s+', header=None, names=cols
    )
    test = pd.read_csv(
        os.path.join(DATA_DIR, f"test_{dataset}.txt"),
        sep=r'\s+', header=None, names=cols
    )
    rul = pd.read_csv(
        os.path.join(DATA_DIR, f"RUL_{dataset}.txt"),
        sep=r'\s+', header=None, names=["RUL"]
    )

    # Add RUL to training data
    max_cycles = train.groupby("engine_id")["cycle"].max().reset_index()
    max_cycles.columns = ["engine_id","max_cycle"]
    train = train.merge(max_cycles, on="engine_id")
    train["RUL"] = train["max_cycle"] - train["cycle"]
    train.drop("max_cycle", axis=1, inplace=True)

    # Add RUL to test data (last cycle of each engine)
    test_last = test.groupby("engine_id").last().reset_index()
    test_last["RUL"] = rul["RUL"].values

    return train, test_last

# ─────────────────────────────────────────────────────────────────────
# 2. SELECT INFORMATIVE SENSORS
# ─────────────────────────────────────────────────────────────────────

# Sensors with near-zero variance are uninformative (known for FD001)
# We use the subset commonly used in literature
SENSOR_COLS = [f"s_{i}" for i in [2,3,4,7,8,9,11,12,13,14,15,17,20,21]]
OP_COLS     = ["op_1","op_2","op_3"]

# ─────────────────────────────────────────────────────────────────────
# 3. BUILD θ(t) PER ENGINE
# ─────────────────────────────────────────────────────────────────────

WINDOW = 15   # 15 cycle rolling window — engines often only run ~200 cycles

def compute_theta_engine(sensor_vals):
    """
    Compute statistical feature vector from a multi-sensor window.
    sensor_vals: (window_size, n_sensors) array
    Returns: 1D feature vector
    """
    feats = []
    for col in range(sensor_vals.shape[1]):
        w = sensor_vals[:, col]
        if np.std(w) < 1e-10:
            feats.extend([np.mean(w), 0, 0, 0, 0, 0])
            continue
        dw = np.diff(w)
        mu = np.mean(w)
        feats.append(mu)                                    # mean level
        feats.append(np.std(w, ddof=1))                    # std level
        feats.append(np.polyfit(np.arange(len(w)),w,1)[0]) # trend slope
        feats.append(np.mean(dw))                           # mean change
        feats.append(np.std(dw, ddof=1) if len(dw)>1 else 0) # volatility
        ac1 = (np.corrcoef(w[:-1],w[1:])[0,1]
               if len(w)>2 else 0)
        feats.append(ac1 if not np.isnan(ac1) else 0)      # autocorrelation

    return np.array(feats)

def build_engine_trajectory(engine_df):
    """
    Build θ(t), velocity, acceleration, D(t) for one engine.
    Returns DataFrame with all features aligned to each cycle.
    """
    sensor_vals = engine_df[SENSOR_COLS].values
    cycles      = engine_df["cycle"].values
    rul_vals    = engine_df["RUL"].values
    n           = len(engine_df)

    thetas, valid_cycles, valid_rul = [], [], []

    for i in range(WINDOW, n):
        w = sensor_vals[i-WINDOW:i]
        th = compute_theta_engine(w)
        thetas.append(th)
        valid_cycles.append(cycles[i])
        valid_rul.append(rul_vals[i])

    if len(thetas) < 5:
        return None

    thetas = np.array(thetas)

    # Standardise
    scaler = StandardScaler()
    thetas_sc = scaler.fit_transform(thetas)

    # Velocity: v(t) = θ(t) - θ(t-1)
    V = np.diff(thetas_sc, axis=0, prepend=thetas_sc[[0]])
    speed = np.linalg.norm(V, axis=1)

    # Acceleration: a(t) = v(t) - v(t-1)
    A     = np.diff(V, axis=0, prepend=V[[0]])
    accel = np.linalg.norm(A, axis=1)

    # Unit direction
    with np.errstate(invalid='ignore', divide='ignore'):
        V_unit = V / (speed[:,None] + 1e-12)
        V_unit = np.nan_to_num(V_unit)

    # Attractor = mean of HEALTHY portion (first 30% of life)
    n_healthy = max(1, int(0.3 * len(thetas_sc)))
    theta_star = thetas_sc[:n_healthy].mean(axis=0)
    dist_star  = np.linalg.norm(thetas_sc - theta_star, axis=1)

    # Displacement over W=10 cycles
    W_disp = 10
    disp_mag = np.zeros(len(thetas_sc))
    disp_dir = np.zeros((len(thetas_sc), thetas_sc.shape[1]))
    for t in range(W_disp, len(thetas_sc)):
        d = thetas_sc[t] - thetas_sc[t-W_disp]
        disp_mag[t] = np.linalg.norm(d)
        disp_dir[t] = d / (np.linalg.norm(d) + 1e-12)

    # Displacement persistence scores
    pers_scores = np.full(len(thetas_sc), np.nan)
    for t in range(W_disp, len(thetas_sc)-W_disp):
        past = thetas_sc[t]   - thetas_sc[t-W_disp]
        fut  = thetas_sc[t+W_disp] - thetas_sc[t]
        pn, fn = np.linalg.norm(past), np.linalg.norm(fut)
        if pn > 1e-10 and fn > 1e-10:
            pers_scores[t] = np.dot(past/pn, fut/fn)

    # PCA for visualisation
    pca = PCA(n_components=min(3, thetas_sc.shape[1]))
    X_pca = pca.fit_transform(thetas_sc)

    result = pd.DataFrame({
        "cycle":     valid_cycles,
        "RUL":       valid_rul,
        "speed":     speed,
        "accel":     accel,
        "dist_star": dist_star,
        "disp_mag":  disp_mag,
        "pers":      pers_scores,
        "PC1":       X_pca[:,0],
        "PC2":       X_pca[:,1],
        "PC3":       X_pca[:,2] if X_pca.shape[1]>2 else 0,
    })

    # Add direction components (PC1,PC2 of velocity)
    pca_v = PCA(n_components=2)
    V_pca = pca_v.fit_transform(V)
    result["vel_PC1"] = V_pca[:,0]
    result["vel_PC2"] = V_pca[:,1]

    return result, thetas_sc, V_unit

# ─────────────────────────────────────────────────────────────────────
# 4. MAIN PROCESSING LOOP
# ─────────────────────────────────────────────────────────────────────

print("="*65)
print("NASA CMAPSS — SMA FRAMEWORK + KINEMATIC FEATURE TEST")
print("="*65)

if not download_cmapss():
    print("\nPlease download data manually and re-run.")
    exit()

# Use FD001 first (single operating condition, clearest signal)
print("\n[1/4] Loading FD001...")
train_df, test_df = load_cmapss("FD001")
engine_ids = train_df["engine_id"].unique()
print(f"  {len(engine_ids)} training engines, "
      f"{len(train_df)} total cycles")

print("\n[2/4] Building θ(t) trajectories per engine...")
all_feats = []
all_rul   = []
all_engine= []
pers_all  = []
dist_all  = []
traj_sample = {}   # store a few for plotting

for eid in engine_ids:
    eng = train_df[train_df["engine_id"]==eid].sort_values("cycle")
    out = build_engine_trajectory(eng)
    if out is None:
        continue
    result, thetas_sc, V_unit = out

    # Store trajectory for plotting (first 5 engines)
    if eid <= 5:
        traj_sample[eid] = result

    # Feature vector for prediction: kinematic features only
    feat_cols = ["speed","accel","dist_star","disp_mag",
                 "vel_PC1","vel_PC2","PC1","PC2","PC3"]
    X = result[feat_cols].fillna(0).values
    y = result["RUL"].values

    all_feats.append(X)
    all_rul.append(y)
    all_engine.extend([eid]*len(y))

    # Collect persistence scores
    ps = result["pers"].dropna().values
    pers_all.extend(ps)
    dist_all.extend(result["dist_star"].values)

all_X  = np.vstack(all_feats)
all_y  = np.concatenate(all_rul)
all_eng= np.array(all_engine)
pers_all = np.array(pers_all)
dist_all = np.array(dist_all)

print(f"  Total samples: {len(all_y)}")
print(f"  Feature dims:  {all_X.shape[1]}")

# ─────────────────────────────────────────────────────────────────────
# 5. DISPLACEMENT PERSISTENCE TEST
# ─────────────────────────────────────────────────────────────────────

print("\n[3/4] Running theoretical tests...")

pers_clean = pers_all[~np.isnan(pers_all)]
t_stat, p_val = ttest_1samp(pers_clean, 0)
dir_acc = (pers_clean > 0).mean()

print(f"\n  Displacement Persistence (W=10 cycles):")
print(f"  n        = {len(pers_clean)}")
print(f"  mean P   = {pers_clean.mean():+.5f}")
print(f"  dir_acc  = {dir_acc:.4f}")
print(f"  t-stat   = {t_stat:.2f}")
print(f"  p-value  = {p_val:.6f}")
sig = "***" if p_val<0.001 else ("**" if p_val<0.01 else
      ("*" if p_val<0.05 else "ns"))
direction = ("MOMENTUM" if pers_clean.mean()>0.01 else
             ("REVERSAL" if pers_clean.mean()<-0.01 else "NEUTRAL"))
print(f"  Result   = {direction} ({sig})")
print(f"\n  Interpretation:")
if pers_clean.mean() > 0.01:
    print("  MOMENTUM confirmed — degradation drifts AWAY from attractor.")
    print("  Directional prediction IS possible in this domain.")
    print("  This is the key result: endogenous drift = predictable direction.")
elif pers_clean.mean() < -0.01:
    print("  REVERSAL — surprising for degradation. Check attractor definition.")
else:
    print("  NEUTRAL — no directional memory.")

# Correlation of kinematic features with RUL
print(f"\n  Spearman correlations with RUL:")
feat_names = ["speed","accel","dist_star","disp_mag",
              "vel_PC1","vel_PC2","PC1","PC2","PC3"]
for i, fname in enumerate(feat_names):
    r, p = spearmanr(all_X[:,i], all_y)
    print(f"    {fname:<15} r={r:+.4f}  p={p:.4f}")

# ─────────────────────────────────────────────────────────────────────
# 6. PREDICTION BASELINE: KINEMATIC VS RAW SENSORS
# ─────────────────────────────────────────────────────────────────────

print("\n[4/4] Training prediction models...")

# Also build raw sensor features for same samples
raw_feats = []
for eid in engine_ids:
    eng = train_df[train_df["engine_id"]==eid].sort_values("cycle")
    if len(eng) <= WINDOW:
        continue
    raw_feats.append(eng[SENSOR_COLS].values[WINDOW:])

raw_X = np.vstack(raw_feats)

# Align sizes
min_len = min(len(raw_X), len(all_X))
raw_X  = raw_X[:min_len]
kin_X  = all_X[:min_len]
y_     = all_y[:min_len]
eng_   = all_eng[:min_len]

# Combined: raw sensors + kinematic features
comb_X = np.hstack([raw_X, kin_X])

# Cross-validation: leave-engines-out
splitter = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

results_cv = {"raw": [], "kinematic": [], "combined": []}

for fold, (train_idx, test_idx) in enumerate(
    splitter.split(raw_X, y_, groups=eng_)
):
    for name, X in [("raw", raw_X),
                    ("kinematic", kin_X),
                    ("combined", comb_X)]:
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[train_idx])
        Xte = scaler.transform(X[test_idx])
        ytr = y_[train_idx]
        yte = y_[test_idx]

        model = GradientBoostingRegressor(
            n_estimators=100, max_depth=4,
            learning_rate=0.1, random_state=42
        )
        model.fit(Xtr, ytr)
        pred = model.predict(Xte)
        rmse = np.sqrt(mean_squared_error(yte, pred))
        results_cv[name].append(rmse)

print("\n  Cross-validated RMSE (5-fold, leave-engines-out):")
print(f"  {'Model':<15} {'Mean RMSE':>10} {'Std':>8}")
print("  " + "-"*35)
for name, scores in results_cv.items():
    print(f"  {name:<15} {np.mean(scores):>10.2f} "
          f"{np.std(scores):>8.2f}")
print(f"\n  Kinematic improvement over raw: "
      f"{np.mean(results_cv['raw'])-np.mean(results_cv['kinematic']):+.2f} RMSE")
print(f"  Combined improvement over raw:  "
      f"{np.mean(results_cv['raw'])-np.mean(results_cv['combined']):+.2f} RMSE")

# Feature importance from combined model
scaler_f = StandardScaler()
comb_sc  = scaler_f.fit_transform(comb_X)
final_model = GradientBoostingRegressor(
    n_estimators=200, max_depth=4,
    learning_rate=0.05, random_state=42
)
final_model.fit(comb_sc, y_)
importances = final_model.feature_importances_
feat_all_names = (SENSOR_COLS +
                  ["speed","accel","dist_star","disp_mag",
                   "vel_PC1","vel_PC2","PC1","PC2","PC3"])

print(f"\n  Top 10 features by importance:")
top_idx = np.argsort(importances)[::-1][:10]
for rank, i in enumerate(top_idx):
    tag = " <-- KINEMATIC" if i >= len(SENSOR_COLS) else ""
    print(f"    {rank+1:>2}. {feat_all_names[i]:<15} "
          f"{importances[i]:.4f}{tag}")

# ─────────────────────────────────────────────────────────────────────
# 7. FIGURES
# ─────────────────────────────────────────────────────────────────────

print("\nGenerating figures...")

# Figure 1: Sample engine trajectories
fig1, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=False)
fig1.suptitle("SMA Framework on CMAPSS Engine Degradation\n"
              "θ(t) kinematics vs Remaining Useful Life",
              fontsize=12, fontweight="bold")

colors = ["#4a7fa5","#c0604a","#7aaa6a","#534AB7","#BA7517"]
for (eid, traj), col in zip(list(traj_sample.items())[:5], colors):
    t   = traj["RUL"].values[::-1]   # flip: 0=start, max=end
    rul = traj["RUL"].values

    axes[0].plot(rul[::-1], traj["speed"].values, lw=0.8,
                 color=col, alpha=0.7, label=f"Eng {eid}")
    axes[1].plot(rul[::-1], traj["dist_star"].values, lw=0.8,
                 color=col, alpha=0.7)
    axes[2].plot(rul[::-1], traj["disp_mag"].values, lw=0.8,
                 color=col, alpha=0.7)
    axes[3].plot(rul[::-1], traj["PC1"].values, lw=0.8,
                 color=col, alpha=0.7)

for ax, ylabel, title in zip(
    axes,
    ["Speed ‖Δθ‖", "Attractor dist D(t)", "Displacement mag", "PC1"],
    ["A. Speed — accelerates near failure?",
     "B. Attractor distance — rises monotonically?",
     "C. Displacement magnitude — grows as degrades?",
     "D. PC1 — consistent drift direction?"]
):
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.spines[["top","right"]].set_visible(False)
    ax.axvline(0, color="#c04040", lw=0.7, ls="--", alpha=0.5)

axes[3].set_xlabel("Cycles since start (RUL decreasing left→right)",
                   fontsize=8)
axes[0].legend(fontsize=7, ncol=5, loc="upper left")
fig1.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/cmapss_fig1_trajectories.png",
            dpi=150, bbox_inches="tight")

# Figure 2: PCA scatter coloured by RUL
fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5))
fig2.suptitle("θ(t) Manifold Geometry Coloured by RUL\n"
              "If geometry encodes degradation, RUL gradient should be visible",
              fontsize=11, fontweight="bold")

all_PC1, all_PC2, all_PC3 = [], [], []
all_rul_plot = []
for eid in list(engine_ids)[:30]:
    eng = train_df[train_df["engine_id"]==eid].sort_values("cycle")
    out = build_engine_trajectory(eng)
    if out is None: continue
    result, _, _ = out
    all_PC1.extend(result["PC1"].values)
    all_PC2.extend(result["PC2"].values)
    all_PC3.extend(result["PC3"].values)
    all_rul_plot.extend(result["RUL"].values)

all_PC1 = np.array(all_PC1)
all_PC2 = np.array(all_PC2)
all_rul_plot = np.array(all_rul_plot)

sc0 = axes2[0].scatter(all_PC1, all_PC2,
                        c=all_rul_plot, cmap="RdYlGn",
                        s=4, alpha=0.4)
plt.colorbar(sc0, ax=axes2[0], label="RUL")
axes2[0].set_xlabel("PC1"); axes2[0].set_ylabel("PC2")
axes2[0].set_title("PC1 vs PC2\n(green=healthy, red=near failure)",
                   fontsize=9)
axes2[0].spines[["top","right"]].set_visible(False)

# RUL bins
bins = [0,25,50,100,200,500]
labels_b = ["0-25","25-50","50-100","100-200","200+"]
colors_b = ["#c04040","#e07020","#e0c020","#70a040","#4070c0"]
for i in range(len(bins)-1):
    mask = (all_rul_plot >= bins[i]) & (all_rul_plot < bins[i+1])
    axes2[1].scatter(all_PC1[mask], all_PC2[mask],
                     s=4, alpha=0.5, color=colors_b[i],
                     label=f"RUL {labels_b[i]}")
axes2[1].set_xlabel("PC1"); axes2[1].set_ylabel("PC2")
axes2[1].set_title("Regime regions by RUL bucket", fontsize=9)
axes2[1].legend(fontsize=6, markerscale=2)
axes2[1].spines[["top","right"]].set_visible(False)

# Dist_star vs RUL
all_dist_plot = []
for eid in list(engine_ids)[:30]:
    eng = train_df[train_df["engine_id"]==eid].sort_values("cycle")
    out = build_engine_trajectory(eng)
    if out is None: continue
    result, _, _ = out
    all_dist_plot.extend(result["dist_star"].values)

all_dist_plot = np.array(all_dist_plot)
axes2[2].scatter(all_rul_plot[::5], all_dist_plot[::5],
                 s=3, alpha=0.2, color="#534AB7")
# Bin mean
rul_bins = np.percentile(all_rul_plot, np.linspace(0,100,20))
bm_rul, bm_dist = [], []
for i in range(len(rul_bins)-1):
    mask = (all_rul_plot>=rul_bins[i]) & (all_rul_plot<rul_bins[i+1])
    if mask.sum()>5:
        bm_rul.append(all_rul_plot[mask].mean())
        bm_dist.append(all_dist_plot[mask].mean())
axes2[2].plot(bm_rul, bm_dist, "o-", color="#c0604a", lw=2, markersize=5)
r_d, p_d = spearmanr(all_rul_plot, all_dist_plot)
axes2[2].set_xlabel("RUL (cycles remaining)"); 
axes2[2].set_ylabel("Attractor distance D(t)")
axes2[2].set_title(f"D(t) vs RUL — r={r_d:.3f}\n"
                   f"(should be negative: less RUL = more stressed)",
                   fontsize=9)
axes2[2].spines[["top","right"]].set_visible(False)

fig2.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/cmapss_fig2_manifold.png",
            dpi=150, bbox_inches="tight")

# Figure 3: Feature importance
fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle("Feature Importance for RUL Prediction\n"
              "Do kinematic features (velocity, direction, distance) add value?",
              fontsize=11, fontweight="bold")

# Left: all features ranked
top20 = np.argsort(importances)[::-1][:20]
cols3 = ["#7aaa6a" if i>=len(SENSOR_COLS) else "#4a7fa5"
         for i in top20]
axes3[0].barh([feat_all_names[i] for i in top20[::-1]],
              importances[top20[::-1]],
              color=cols3[::-1], alpha=0.85)
axes3[0].set_xlabel("Feature importance", fontsize=9)
axes3[0].set_title("All features ranked\n(green = kinematic, blue = raw sensor)",
                   fontsize=9)
axes3[0].spines[["top","right"]].set_visible(False)
from matplotlib.patches import Patch
axes3[0].legend(handles=[
    Patch(facecolor="#7aaa6a", label="Kinematic (SMA)"),
    Patch(facecolor="#4a7fa5", label="Raw sensor"),
], fontsize=8)

# Right: RMSE comparison
model_names = ["Raw sensors\nonly",
               "Kinematic\nfeatures only",
               "Combined\n(raw + kinematic)"]
rmse_means  = [np.mean(results_cv[k])
               for k in ["raw","kinematic","combined"]]
rmse_stds   = [np.std(results_cv[k])
               for k in ["raw","kinematic","combined"]]
bar_colors  = ["#4a7fa5","#7aaa6a","#534AB7"]
bars = axes3[1].bar(model_names, rmse_means, yerr=rmse_stds,
                    color=bar_colors, alpha=0.85, capsize=5)
for bar, m, s in zip(bars, rmse_means, rmse_stds):
    axes3[1].text(bar.get_x()+bar.get_width()/2,
                  m+s+0.5, f"{m:.1f}", ha="center",
                  fontsize=9, fontweight="bold")
axes3[1].set_ylabel("RMSE (cycles)", fontsize=9)
axes3[1].set_title("RUL prediction RMSE\n(lower is better)", fontsize=9)
axes3[1].spines[["top","right"]].set_visible(False)

fig3.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/cmapss_fig3_importance.png",
            dpi=150, bbox_inches="tight")

# Figure 4: Displacement persistence + direction test
fig4, axes4 = plt.subplots(1, 3, figsize=(16, 5))
fig4.suptitle("Directional Test: Is Degradation Drift Directional?\n"
              "E[P(t,W)] > 0 = MOMENTUM = directional prediction works here",
              fontsize=11, fontweight="bold")

# Histogram of persistence scores
axes4[0].hist(pers_clean, bins=60, color="#7aaa6a", alpha=0.8, density=True)
axes4[0].axvline(0, color="#999", lw=1.5, ls="--", label="Zero (random walk)")
axes4[0].axvline(pers_clean.mean(), color="#2a6e2a", lw=2,
                 label=f"Mean={pers_clean.mean():+.4f}")
col_res = "#7aaa6a" if pers_clean.mean()>0 else "#c0604a"
axes4[0].set_title(
    f"Displacement persistence distribution\n"
    f"→ {'MOMENTUM' if pers_clean.mean()>0.01 else 'REVERSAL/NEUTRAL'} "
    f"(t={t_stat:.1f}, p={p_val:.4f}, {sig})",
    fontsize=9,
    color="#2a6e2a" if pers_clean.mean()>0.01 else "#c04040"
)
axes4[0].set_xlabel("P(t,W)", fontsize=9)
axes4[0].legend(fontsize=7)
axes4[0].spines[["top","right"]].set_visible(False)

# Speed vs RUL
axes4[1].scatter(all_rul_plot[::5],
                 all_X[:len(all_rul_plot):5, 0],
                 s=3, alpha=0.2, color="#c0604a")
r_s, p_s = spearmanr(all_rul_plot, all_X[:len(all_rul_plot),0])
rul_b = np.percentile(all_rul_plot, np.linspace(0,100,20))
bm_rul2, bm_spd = [], []
speed_all = all_X[:len(all_rul_plot),0]
for i in range(len(rul_b)-1):
    mask = (all_rul_plot>=rul_b[i]) & (all_rul_plot<rul_b[i+1])
    if mask.sum()>5:
        bm_rul2.append(all_rul_plot[mask].mean())
        bm_spd.append(speed_all[mask].mean())
axes4[1].plot(bm_rul2, bm_spd, "o-", color="#934030", lw=2, markersize=5)
axes4[1].set_xlabel("RUL (cycles remaining)", fontsize=9)
axes4[1].set_ylabel("Speed ‖Δθ‖", fontsize=9)
axes4[1].set_title(f"Speed vs RUL (r={r_s:.3f})\n"
                   f"(should rise as RUL→0)", fontsize=9)
axes4[1].spines[["top","right"]].set_visible(False)

# Persistence by RUL quartile
rul_q = np.percentile(all_rul_plot[:len(pers_all)],
                      [0,25,50,75,100])
quartile_labels = ["Q1\n(near failure)","Q2","Q3","Q4\n(healthy)"]
quartile_pers   = []
for i in range(4):
    mask = ((all_rul_plot[:len(pers_all)] >= rul_q[i]) &
            (all_rul_plot[:len(pers_all)] < rul_q[i+1]))
    ps = pers_all[mask]
    ps = ps[~np.isnan(ps)]
    quartile_pers.append(ps.mean() if len(ps)>0 else 0)

bar_c = ["#c04040" if p>0 else "#4a7fa5" for p in quartile_pers]
axes4[2].bar(quartile_labels, quartile_pers, color=bar_c, alpha=0.85)
axes4[2].axhline(0, color="#999", lw=1.5, ls="--")
axes4[2].set_ylabel("Mean persistence E[P(t,W)]", fontsize=9)
axes4[2].set_title("Persistence by RUL quartile\n"
                   "(>0 near failure = momentum near end of life)", fontsize=9)
axes4[2].spines[["top","right"]].set_visible(False)

fig4.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/cmapss_fig4_direction.png",
            dpi=150, bbox_inches="tight")

print("  Saved 4 figures to outputs/")

# ─────────────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────

print("\n" + "="*65)
print("CMAPSS RESULTS SUMMARY")
print("="*65)
print(f"\n  Displacement persistence: {pers_clean.mean():+.5f} ({sig})")
print(f"  Directional accuracy:     {dir_acc:.4f}")
print(f"  → {'ENDOGENOUS MOMENTUM CONFIRMED' if pers_clean.mean()>0.01 else 'NOT CONFIRMED'}")
print(f"\n  Attractor dist vs RUL:    r={r_d:.4f}")
print(f"  Speed vs RUL:             r={r_s:.4f}")
print(f"\n  Prediction RMSE:")
for k in ["raw","kinematic","combined"]:
    print(f"    {k:<12}: {np.mean(results_cv[k]):.2f} ± "
          f"{np.std(results_cv[k]):.2f}")
print(f"\n  Key finding:")
if pers_clean.mean() > 0.01:
    print("  MOMENTUM: degradation IS directional in θ-space.")
    print("  Velocity and direction of θ(t) carry predictive signal.")
    print("  → RWKV model on kinematic features is justified.")
else:
    print("  Direction hypothesis not confirmed on this dataset.")
    print("  Check attractor definition or window size.")


NASA CMAPSS — SMA FRAMEWORK + KINEMATIC FEATURE TEST
  Download failed: HTTP Error 404: Not Found
  Manual download: go to https://data.nasa.gov/dataset/Turbofan-Engine-Degradation-Simulation-Data-Set-1/vrks-gjie
  Place train_FD001.txt ... test_FD004.txt in cmapss_data/

Please download data manually and re-run.

[1/4] Loading FD001...


FileNotFoundError: [Errno 2] No such file or directory: 'cmapss_data\\train_FD001.txt'

: 

In [2]:
"""
NASA CMAPSS — SMA Framework Analysis
=====================================
Run this on your local PC.

STEP 1: Install dependencies
    pip install numpy pandas matplotlib scikit-learn scipy

STEP 2: Download data (one time)
    Go to: https://data.nasa.gov/dataset/Turbofan-Engine-Degradation-Simulation-Data-Set-1/vrks-gjie
    Download the ZIP file → extract to a folder called  cmapss_data/
    You need these files inside cmapss_data/:
        train_FD001.txt
        test_FD001.txt
        RUL_FD001.txt
        (FD002, FD003, FD004 optional)

    OR just run this script — it will try to auto-download.

STEP 3: Run
    python cmapss_analysis.py
"""

import warnings
warnings.filterwarnings("ignore")
import os, sys, zipfile, urllib.request

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # works headless; change to "TkAgg" if you want popup windows
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis, ttest_1samp, spearmanr
from scipy.spatial.distance import cosine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from matplotlib.patches import Patch

# ──────────────────────────────────────────────────────────────────────────
# CONFIG  — change these if needed
# ──────────────────────────────────────────────────────────────────────────

DATA_DIR   = "cmapss_data"          # where the .txt files live
OUTPUT_DIR = "cmapss_outputs"       # where plots are saved
DATASET    = "FD001"                # FD001 / FD002 / FD003 / FD004
WINDOW     = 15                     # rolling window for θ(t)  (cycles)
W_DISP     = 10                     # displacement window      (cycles)

# Sensors that are actually informative in FD001 (removes constant ones)
SENSOR_COLS = [f"s_{i}" for i in [2,3,4,7,8,9,11,12,13,14,15,17,20,21]]

os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ──────────────────────────────────────────────────────────────────────────
# 1.  DOWNLOAD DATA
# ──────────────────────────────────────────────────────────────────────────

def download_cmapss():
    target = os.path.join(DATA_DIR, "train_FD001.txt")
    if os.path.exists(target):
        print("  Data already present — skipping download.")
        return True

    # Mirror 1: GitHub (small, reliable)
    mirrors = [
        ("https://raw.githubusercontent.com/schwingbach/"
         "deepl-rul-cmapss/master/CMAPSSData.zip"),
        ("https://raw.githubusercontent.com/Azure/lstms_for_predictive_"
         "maintenance/master/Deep%20Learning%20Basics%20for%20Predictive%20"
         "Maintenance.ipynb"),   # won't work but gives a clear 404
    ]

    zip_path = os.path.join(DATA_DIR, "CMAPSSData.zip")
    for url in mirrors[:1]:
        try:
            print(f"  Downloading from {url[:60]}...")
            urllib.request.urlretrieve(url, zip_path)
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(DATA_DIR)
            print("  Download complete.")
            return True
        except Exception as e:
            print(f"  Failed: {e}")

    print("""
  ──────────────────────────────────────────────────────────────
  AUTO-DOWNLOAD FAILED.  Manual steps (takes 2 minutes):

  1. Open:  https://data.nasa.gov/dataset/
             Turbofan-Engine-Degradation-Simulation-Data-Set-1/vrks-gjie
  2. Click "Download" → save the ZIP
  3. Extract → you'll get a folder with .txt files
  4. Copy these into your  cmapss_data/  folder:
       train_FD001.txt   test_FD001.txt   RUL_FD001.txt
  5. Re-run this script
  ──────────────────────────────────────────────────────────────
    """)
    return False


# ──────────────────────────────────────────────────────────────────────────
# 2.  LOAD DATA
# ──────────────────────────────────────────────────────────────────────────

def load_cmapss(dataset):
    cols = (["engine_id", "cycle"]
            + [f"op_{i}" for i in range(1, 4)]
            + [f"s_{i}"  for i in range(1, 22)])

    def read(fname):
        return pd.read_csv(
            os.path.join(DATA_DIR, fname),
            sep=r"\s+", header=None, names=cols
        )

    train = read(f"train_{dataset}.txt")
    test  = read(f"test_{dataset}.txt")
    rul   = pd.read_csv(
        os.path.join(DATA_DIR, f"RUL_{dataset}.txt"),
        sep=r"\s+", header=None, names=["RUL"]
    )

    # Add RUL to training rows
    mx = train.groupby("engine_id")["cycle"].max().rename("max_cycle")
    train = train.join(mx, on="engine_id")
    train["RUL"] = train["max_cycle"] - train["cycle"]
    train.drop("max_cycle", axis=1, inplace=True)

    return train, test, rul


# ──────────────────────────────────────────────────────────────────────────
# 3.  BUILD θ(t) FOR ONE ENGINE
# ──────────────────────────────────────────────────────────────────────────

def theta_window(sensor_window):
    """
    sensor_window : (WINDOW, n_sensors) ndarray
    returns       : 1-D feature vector
    """
    feats = []
    for j in range(sensor_window.shape[1]):
        w  = sensor_window[:, j]
        dw = np.diff(w)
        if np.std(w) < 1e-10:               # constant sensor → zeros
            feats += [np.mean(w), 0, 0, 0, 0, 0]
            continue
        feats.append(np.mean(w))                                  # level
        feats.append(np.std(w, ddof=1))                           # volatility
        feats.append(np.polyfit(np.arange(len(w)), w, 1)[0])     # trend
        feats.append(np.mean(dw))                                 # mean change
        feats.append(np.std(dw, ddof=1) if len(dw) > 1 else 0)   # change vol
        ac = (np.corrcoef(w[:-1], w[1:])[0, 1]
              if len(w) > 2 else 0.0)
        feats.append(float(ac) if not np.isnan(ac) else 0.0)     # autocorr
    return np.array(feats, dtype=np.float32)


def engine_features(eng_df):
    """
    Returns a DataFrame with θ-kinematics for one engine,
    or None if the engine is too short.
    """
    sv   = eng_df[SENSOR_COLS].values.astype(np.float32)
    rul  = eng_df["RUL"].values
    cyc  = eng_df["cycle"].values
    n    = len(eng_df)

    if n <= WINDOW + W_DISP + 2:
        return None, None, None

    # ── build raw θ matrix ─────────────────────────────────────────
    thetas = np.array([
        theta_window(sv[i - WINDOW: i])
        for i in range(WINDOW, n)
    ])

    # standardise within this engine
    sc      = StandardScaler()
    thetas  = sc.fit_transform(thetas)

    # ── velocity, speed, acceleration ─────────────────────────────
    V     = np.diff(thetas, axis=0, prepend=thetas[[0]])
    speed = np.linalg.norm(V, axis=1)
    A     = np.diff(V,      axis=0, prepend=V[[0]])
    accel = np.linalg.norm(A, axis=1)

    with np.errstate(invalid="ignore", divide="ignore"):
        V_unit = V / (speed[:, None] + 1e-12)
        V_unit = np.nan_to_num(V_unit)

    # ── attractor  (healthy = first 30 % of life) ─────────────────
    n_h       = max(1, int(0.30 * len(thetas)))
    theta_star = thetas[:n_h].mean(axis=0)
    dist_star  = np.linalg.norm(thetas - theta_star, axis=1)

    # ── displacement  ──────────────────────────────────────────────
    disp_mag = np.zeros(len(thetas))
    for t in range(W_DISP, len(thetas)):
        disp_mag[t] = np.linalg.norm(thetas[t] - thetas[t - W_DISP])

    # ── displacement persistence scores ───────────────────────────
    pers = np.full(len(thetas), np.nan)
    for t in range(W_DISP, len(thetas) - W_DISP):
        past = thetas[t]          - thetas[t - W_DISP]
        fut  = thetas[t + W_DISP] - thetas[t]
        pn, fn = np.linalg.norm(past), np.linalg.norm(fut)
        if pn > 1e-10 and fn > 1e-10:
            pers[t] = float(np.dot(past / pn, fut / fn))

    # ── PCA (2-D) for visualisation ───────────────────────────────
    pca   = PCA(n_components=min(3, thetas.shape[1]))
    X_pca = pca.fit_transform(thetas)

    # ── velocity PCA (2-D direction summary) ──────────────────────
    pca_v  = PCA(n_components=2)
    V_2d   = pca_v.fit_transform(V)

    row_rul = rul[WINDOW:]
    row_cyc = cyc[WINDOW:]

    df = pd.DataFrame({
        "cycle":     row_cyc,
        "RUL":       row_rul,
        "speed":     speed,
        "accel":     accel,
        "dist_star": dist_star,
        "disp_mag":  disp_mag,
        "pers":      pers,
        "PC1":       X_pca[:, 0],
        "PC2":       X_pca[:, 1],
        "PC3":       X_pca[:, 2] if X_pca.shape[1] > 2 else 0.0,
        "vel_PC1":   V_2d[:, 0],
        "vel_PC2":   V_2d[:, 1],
    })
    return df, thetas, V_unit


# ──────────────────────────────────────────────────────────────────────────
# 4.  PROCESS ALL ENGINES
# ──────────────────────────────────────────────────────────────────────────

print("=" * 65)
print("NASA CMAPSS — SMA FRAMEWORK + KINEMATIC FEATURE TEST")
print("=" * 65)

if not download_cmapss():
    sys.exit(1)

print(f"\n[1/5]  Loading {DATASET} ...")
train_df, test_df, rul_df = load_cmapss(DATASET)
engine_ids = sorted(train_df["engine_id"].unique())
print(f"       {len(engine_ids)} engines,  "
      f"{len(train_df):,} total rows")

print(f"\n[2/5]  Building θ(t) per engine  (window={WINDOW} cycles) ...")

KIN_COLS  = ["speed", "accel", "dist_star", "disp_mag",
             "vel_PC1", "vel_PC2", "PC1", "PC2", "PC3"]

all_kin,   all_raw,  all_rul, all_eng = [], [], [], []
pers_pool  = []
dist_pool  = []
rul_pool   = []
traj_plot  = {}      # first 6 engines for plotting

for eid in engine_ids:
    sub = train_df[train_df["engine_id"] == eid].sort_values("cycle")
    df, thetas, _ = engine_features(sub)
    if df is None:
        continue

    if eid <= 6:
        traj_plot[eid] = df

    X_kin = df[KIN_COLS].fillna(0).values
    X_raw = sub[SENSOR_COLS].values[WINDOW: WINDOW + len(df)]
    y     = df["RUL"].values

    min_len = min(len(X_kin), len(X_raw), len(y))
    all_kin.append(X_kin[:min_len])
    all_raw.append(X_raw[:min_len])
    all_rul.append(y[:min_len])
    all_eng.extend([eid] * min_len)

    ps = df["pers"].dropna().values
    pers_pool.extend(ps)
    dist_pool.extend(df["dist_star"].values)
    rul_pool.extend(df["RUL"].values)

all_kin  = np.vstack(all_kin)
all_raw  = np.vstack(all_raw)
all_rul  = np.concatenate(all_rul)
all_eng  = np.array(all_eng)
pers_arr = np.array(pers_pool)
dist_arr = np.array(dist_pool)
rul_arr  = np.array(rul_pool)

print(f"       Total samples : {len(all_rul):,}")
print(f"       Kinematic dims: {all_kin.shape[1]}")
print(f"       Raw sensor dims: {all_raw.shape[1]}")


# ──────────────────────────────────────────────────────────────────────────
# 5.  THEORETICAL TESTS
# ──────────────────────────────────────────────────────────────────────────

print(f"\n[3/5]  Theoretical tests ...")

pers_clean = pers_arr[~np.isnan(pers_arr)]
t_stat, p_val = ttest_1samp(pers_clean, 0)
dir_acc = (pers_clean > 0).mean()
sig = ("***" if p_val < 0.001 else
       "**"  if p_val < 0.01  else
       "*"   if p_val < 0.05  else "ns")
direction = ("MOMENTUM" if pers_clean.mean() > 0.01 else
             "REVERSAL" if pers_clean.mean() < -0.01 else "NEUTRAL")

print(f"\n  Displacement persistence  (W={W_DISP} cycles):")
print(f"    n           = {len(pers_clean):,}")
print(f"    mean P(t,W) = {pers_clean.mean():+.5f}")
print(f"    dir_acc     = {dir_acc:.4f}   (> 0.5 = momentum)")
print(f"    t-stat      = {t_stat:+.2f}")
print(f"    p-value     = {p_val:.2e}  {sig}")
print(f"    → {direction}")
if direction == "MOMENTUM":
    print("    ✓ Degradation IS directional — RWKV on θ-kinematics is justified.")
elif direction == "REVERSAL":
    print("    ✗ Mean reversion — same as financial data.")
else:
    print("    ~ No clear directional memory.")

print(f"\n  Spearman correlations with RUL:")
for i, name in enumerate(KIN_COLS):
    r, p = spearmanr(all_kin[:, i], all_rul)
    star = "***" if p < 0.001 else ("**" if p < 0.01 else
           ("*"   if p < 0.05 else ""))
    print(f"    {name:<15}  r = {r:+.4f}  {star}")

r_dist, _ = spearmanr(dist_arr, rul_arr)
r_spd,  _ = spearmanr(all_kin[:len(rul_arr), 0], rul_arr)
print(f"\n  Attractor dist vs RUL:  r = {r_dist:+.4f}")
print(f"  Speed vs RUL:           r = {r_spd:+.4f}")


# ──────────────────────────────────────────────────────────────────────────
# 6.  PREDICTION COMPARISON
# ──────────────────────────────────────────────────────────────────────────

print(f"\n[4/5]  Training GBT models (5-fold leave-engines-out) ...")

comb = np.hstack([all_raw, all_kin])
splits = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

cv = {"raw": [], "kinematic": [], "combined": []}

for tr_idx, te_idx in splits.split(all_raw, all_rul, groups=all_eng):
    for tag, X in [("raw", all_raw),
                   ("kinematic", all_kin),
                   ("combined", comb)]:
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X[tr_idx])
        Xte = sc.transform(X[te_idx])
        m   = GradientBoostingRegressor(
                n_estimators=100, max_depth=4,
                learning_rate=0.1, random_state=42)
        m.fit(Xtr, all_rul[tr_idx])
        rmse = np.sqrt(mean_squared_error(
                all_rul[te_idx], m.predict(Xte)))
        cv[tag].append(rmse)

print(f"\n  {'Model':<14}  {'Mean RMSE':>10}  {'±Std':>8}")
print("  " + "-" * 36)
for tag in ["raw", "kinematic", "combined"]:
    print(f"  {tag:<14}  {np.mean(cv[tag]):>10.2f}  "
          f"{np.std(cv[tag]):>8.2f}")

imp_raw  = np.mean(cv["raw"])
imp_kin  = np.mean(cv["kinematic"])
imp_comb = np.mean(cv["combined"])
print(f"\n  Kinematic vs raw:   {imp_raw - imp_kin:+.2f} RMSE "
      f"({'better' if imp_kin < imp_raw else 'worse'})")
print(f"  Combined vs raw:    {imp_raw - imp_comb:+.2f} RMSE "
      f"({'better' if imp_comb < imp_raw else 'worse'})")

# final model for importance
sc_f   = StandardScaler()
comb_f = sc_f.fit_transform(comb)
m_f    = GradientBoostingRegressor(
            n_estimators=200, max_depth=4,
            learning_rate=0.05, random_state=42)
m_f.fit(comb_f, all_rul)
imps  = m_f.feature_importances_
names = SENSOR_COLS + KIN_COLS
top15 = np.argsort(imps)[::-1][:15]

print(f"\n  Top 15 features:")
for rank, i in enumerate(top15):
    tag = " ← KINEMATIC" if i >= len(SENSOR_COLS) else ""
    print(f"    {rank+1:>2}.  {names[i]:<16}  {imps[i]:.4f}{tag}")


# ──────────────────────────────────────────────────────────────────────────
# 7.  FIGURES
# ──────────────────────────────────────────────────────────────────────────

print(f"\n[5/5]  Saving figures to  {OUTPUT_DIR}/  ...")
PAL = ["#4a7fa5","#c0604a","#7aaa6a","#534AB7","#BA7517","#a04060"]

# ── FIG 1 : per-engine kinematic profiles ────────────────────────────────
fig1, axs = plt.subplots(4, 1, figsize=(14, 11), sharex=False)
fig1.suptitle("SMA Framework — CMAPSS Engine Degradation\n"
              "θ(t) kinematics over engine life (x-axis = cycle, 0 = new)",
              fontsize=12, fontweight="bold")

panels = [
    ("speed",     "Speed ‖Δθ‖",          "A. Speed"),
    ("dist_star", "Attractor distance D(t)", "B. Attractor distance"),
    ("disp_mag",  "Displacement ‖d(t,W)‖",  "C. Displacement magnitude"),
    ("PC1",       "PC1",                  "D. PC1 trajectory"),
]
for ax, (col, ylabel, title) in zip(axs, panels):
    for (eid, traj), c in zip(list(traj_plot.items())[:6], PAL):
        x = traj["cycle"].values
        ax.plot(x, traj[col].values, lw=0.9, color=c,
                alpha=0.75, label=f"E{eid}")
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.spines[["top","right"]].set_visible(False)

axs[0].legend(fontsize=7, ncol=6, loc="upper left")
axs[-1].set_xlabel("Cycle", fontsize=9)
fig1.tight_layout()
fig1.savefig(f"{OUTPUT_DIR}/fig1_trajectories.png",
             dpi=150, bbox_inches="tight")
plt.close(fig1)

# ── FIG 2 : PCA manifold coloured by RUL ─────────────────────────────────
# Collect PCA coords + RUL for first 40 engines
pc1_v, pc2_v, rul_v, dist_v = [], [], [], []
for eid in engine_ids[:40]:
    sub = train_df[train_df["engine_id"]==eid].sort_values("cycle")
    df, _, _ = engine_features(sub)
    if df is None: continue
    pc1_v.extend(df["PC1"].values)
    pc2_v.extend(df["PC2"].values)
    rul_v.extend(df["RUL"].values)
    dist_v.extend(df["dist_star"].values)

pc1_v = np.array(pc1_v)
pc2_v = np.array(pc2_v)
rul_v = np.array(rul_v)
dist_v= np.array(dist_v)

fig2, axs2 = plt.subplots(1, 3, figsize=(16, 5))
fig2.suptitle("θ(t) Manifold Geometry Coloured by RUL\n"
              "(green = healthy, red = near failure)",
              fontsize=11, fontweight="bold")

# continuous colour
sc = axs2[0].scatter(pc1_v[::3], pc2_v[::3],
                     c=rul_v[::3], cmap="RdYlGn",
                     s=4, alpha=0.4, vmin=0, vmax=rul_v.max())
plt.colorbar(sc, ax=axs2[0], label="RUL (cycles)")
axs2[0].set_xlabel("PC1"); axs2[0].set_ylabel("PC2")
axs2[0].set_title("Continuous RUL colour", fontsize=9)
axs2[0].spines[["top","right"]].set_visible(False)

# bucketed colour
buckets  = [(0,25,"#c04040","0-25"), (25,75,"#e07020","25-75"),
            (75,150,"#c0c020","75-150"), (150,1e9,"#4070c0","150+")]
for lo, hi, c, lbl in buckets:
    m = (rul_v >= lo) & (rul_v < hi)
    axs2[1].scatter(pc1_v[m], pc2_v[m], s=4, alpha=0.45,
                    color=c, label=f"RUL {lbl}")
axs2[1].set_xlabel("PC1"); axs2[1].set_ylabel("PC2")
axs2[1].set_title("RUL buckets", fontsize=9)
axs2[1].legend(fontsize=7, markerscale=2)
axs2[1].spines[["top","right"]].set_visible(False)

# D(t) vs RUL scatter + bin means
axs2[2].scatter(rul_v[::4], dist_v[::4],
                s=3, alpha=0.15, color="#534AB7")
qs = np.percentile(rul_v, np.linspace(0, 100, 20))
bx, by = [], []
for i in range(len(qs)-1):
    msk = (rul_v >= qs[i]) & (rul_v < qs[i+1])
    if msk.sum() > 5:
        bx.append(rul_v[msk].mean())
        by.append(dist_v[msk].mean())
axs2[2].plot(bx, by, "o-", color="#c0604a", lw=2, markersize=5)
r_rv, _ = spearmanr(rul_v, dist_v)
axs2[2].set_xlabel("RUL"); axs2[2].set_ylabel("D(t)")
axs2[2].set_title(f"D(t) vs RUL  (r = {r_rv:.3f})\n"
                  f"Negative = sicker engine = farther from attractor",
                  fontsize=9)
axs2[2].spines[["top","right"]].set_visible(False)

fig2.tight_layout()
fig2.savefig(f"{OUTPUT_DIR}/fig2_manifold.png",
             dpi=150, bbox_inches="tight")
plt.close(fig2)

# ── FIG 3 : feature importance + RMSE ────────────────────────────────────
fig3, axs3 = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle("Feature Importance and Prediction RMSE\n"
              "Do kinematic features (velocity, direction, distance) add signal?",
              fontsize=11, fontweight="bold")

bar_cols = ["#7aaa6a" if i >= len(SENSOR_COLS) else "#4a7fa5"
            for i in top15[::-1]]
axs3[0].barh([names[i] for i in top15[::-1]],
             imps[top15[::-1]],
             color=bar_cols, alpha=0.85)
axs3[0].set_xlabel("Feature importance", fontsize=9)
axs3[0].set_title("Top 15 features", fontsize=9)
axs3[0].legend(handles=[
    Patch(facecolor="#7aaa6a", label="Kinematic (SMA)"),
    Patch(facecolor="#4a7fa5", label="Raw sensor"),
], fontsize=8)
axs3[0].spines[["top","right"]].set_visible(False)

mnames = ["Raw sensors", "Kinematic only", "Combined"]
rmses  = [np.mean(cv[k]) for k in ["raw","kinematic","combined"]]
errs   = [np.std( cv[k]) for k in ["raw","kinematic","combined"]]
b_cols = ["#4a7fa5","#7aaa6a","#534AB7"]
bars   = axs3[1].bar(mnames, rmses, yerr=errs, color=b_cols,
                     alpha=0.85, capsize=5, width=0.5)
for bar, m in zip(bars, rmses):
    axs3[1].text(bar.get_x()+bar.get_width()/2,
                 m+1, f"{m:.1f}", ha="center", fontsize=9,
                 fontweight="bold")
axs3[1].set_ylabel("RMSE (cycles)", fontsize=9)
axs3[1].set_title("RUL prediction RMSE (lower = better)", fontsize=9)
axs3[1].spines[["top","right"]].set_visible(False)

fig3.tight_layout()
fig3.savefig(f"{OUTPUT_DIR}/fig3_importance.png",
             dpi=150, bbox_inches="tight")
plt.close(fig3)

# ── FIG 4 : directional / persistence results ─────────────────────────────
fig4, axs4 = plt.subplots(1, 3, figsize=(16, 5))
fig4.suptitle("Directional Test — Is Degradation Drift Directional?\n"
              "E[P(t,W)] > 0 means the vector keeps moving the same way "
              "→ directional prediction is possible",
              fontsize=11, fontweight="bold")

# persistence histogram
c_hist = "#7aaa6a" if pers_clean.mean() > 0 else "#c0604a"
axs4[0].hist(pers_clean, bins=60, color=c_hist, alpha=0.8, density=True)
axs4[0].axvline(0, color="#888", lw=1.5, ls="--")
axs4[0].axvline(pers_clean.mean(), color="#2a5e2a", lw=2.5,
                label=f"mean = {pers_clean.mean():+.4f}")
axs4[0].set_xlabel("P(t,W)  cosine similarity", fontsize=9)
axs4[0].set_ylabel("Density", fontsize=9)
axs4[0].set_title(
    f"Displacement persistence\n"
    f"→ {direction}   ({sig}, t={t_stat:.1f})",
    fontsize=9,
    color="#2a5e2a" if pers_clean.mean()>0.01 else "#c04040")
axs4[0].legend(fontsize=8)
axs4[0].spines[["top","right"]].set_visible(False)

# speed vs RUL
speed_pool = all_kin[:len(rul_arr), 0]
axs4[1].scatter(rul_arr[::5], speed_pool[::5],
                s=3, alpha=0.2, color="#c0604a")
qs2 = np.percentile(rul_arr, np.linspace(0, 100, 20))
bx2, by2 = [], []
for i in range(len(qs2)-1):
    msk = (rul_arr >= qs2[i]) & (rul_arr < qs2[i+1])
    if msk.sum() > 5:
        bx2.append(rul_arr[msk].mean())
        by2.append(speed_pool[msk].mean())
axs4[1].plot(bx2, by2, "o-", color="#834020", lw=2, markersize=5)
rs2, _ = spearmanr(rul_arr, speed_pool)
axs4[1].set_xlabel("RUL", fontsize=9)
axs4[1].set_ylabel("Speed ‖Δθ‖", fontsize=9)
axs4[1].set_title(f"Speed vs RUL  (r = {rs2:.3f})\n"
                  f"Negative r = speed rises as engine fails", fontsize=9)
axs4[1].spines[["top","right"]].set_visible(False)

# persistence by RUL quartile
rul_for_pers = rul_arr[:len(pers_arr)]
qs3 = np.percentile(rul_for_pers, [0, 25, 50, 75, 100])
q_labels = ["Q1\n(near fail)", "Q2", "Q3", "Q4\n(healthy)"]
q_pers   = []
for i in range(4):
    msk = ((rul_for_pers >= qs3[i]) &
           (rul_for_pers < qs3[i+1]))
    ps  = pers_arr[msk]
    ps  = ps[~np.isnan(ps)]
    q_pers.append(ps.mean() if len(ps) > 0 else 0.0)

bar_c4 = ["#c04040" if p > 0 else "#4a7fa5" for p in q_pers]
axs4[2].bar(q_labels, q_pers, color=bar_c4, alpha=0.85)
axs4[2].axhline(0, color="#888", lw=1.5, ls="--")
axs4[2].set_ylabel("Mean P(t,W)", fontsize=9)
axs4[2].set_title("Persistence by RUL quartile\n"
                  "> 0 near failure = momentum at end of life", fontsize=9)
axs4[2].spines[["top","right"]].set_visible(False)

fig4.tight_layout()
fig4.savefig(f"{OUTPUT_DIR}/fig4_direction.png",
             dpi=150, bbox_inches="tight")
plt.close(fig4)

# ──────────────────────────────────────────────────────────────────────────
# 8.  FINAL SUMMARY
# ──────────────────────────────────────────────────────────────────────────

print("\n" + "="*65)
print("FINAL SUMMARY")
print("="*65)
print(f"\n  Dataset            : CMAPSS {DATASET}")
print(f"  Engines            : {len(engine_ids)}")
print(f"  Total samples      : {len(all_rul):,}")
print(f"\n  Displacement persistence:")
print(f"    mean P(t,W)      = {pers_clean.mean():+.5f}  ({sig})")
print(f"    dir_acc          = {dir_acc:.4f}  "
      f"({'> 0.5 → MOMENTUM' if dir_acc>0.5 else '< 0.5 → REVERSAL'})")
print(f"\n  Correlations:")
print(f"    D(t) vs RUL      = {r_dist:+.4f}")
print(f"    speed vs RUL     = {r_spd:+.4f}")
print(f"\n  Prediction RMSE:")
for tag in ["raw","kinematic","combined"]:
    print(f"    {tag:<12}   = {np.mean(cv[tag]):.2f}")
print(f"\n  Kinematic improvement = "
      f"{np.mean(cv['raw'])-np.mean(cv['combined']):+.2f} cycles RMSE")
print(f"\n  Core finding: {direction}")
if direction == "MOMENTUM":
    print("  → Degradation trajectory IS directional.")
    print("  → Velocity + direction + distance are valid RWKV input features.")
print(f"\n  Figures saved:")
for f in ["fig1_trajectories.png","fig2_manifold.png",
          "fig3_importance.png","fig4_direction.png"]:
    print(f"    {OUTPUT_DIR}/{f}")

NASA CMAPSS — SMA FRAMEWORK + KINEMATIC FEATURE TEST
  Data already present — skipping download.

[1/5]  Loading FD001 ...
       100 engines,  20,631 total rows

[2/5]  Building θ(t) per engine  (window=15 cycles) ...
       Total samples : 19,131
       Kinematic dims: 9
       Raw sensor dims: 14

[3/5]  Theoretical tests ...

  Displacement persistence  (W=10 cycles):
    n           = 17,131
    mean P(t,W) = -0.42861
    dir_acc     = 0.0011   (> 0.5 = momentum)
    t-stat      = -472.32
    p-value     = 0.00e+00  ***
    → REVERSAL
    ✗ Mean reversion — same as financial data.

  Spearman correlations with RUL:
    speed            r = -0.0495  ***
    accel            r = -0.0485  ***
    dist_star        r = -0.8148  ***
    disp_mag         r = -0.2763  ***
    vel_PC1          r = -0.0004  
    vel_PC2          r = -0.0009  
    PC1              r = -0.5814  ***
    PC2              r = +0.0214  **
    PC3              r = +0.0161  *

  Attractor dist vs RUL:  r = -0.8148
